<a href="https://colab.research.google.com/github/HST0077/HYOTC/blob/main/IRS%EC%99%80_Swaption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 목요일에 SOFR로 차입하여 월요일에 상환하는 경우의 이자는 아래의 식으로 계산된다. 이 때, 목요일의 SOFR는 5%, 금요일의 SOFR는 6%라고 가정하자.

In [1]:
(1+0.05*1/360)*(1+0.06*3/360)-1

0.0006389583333332727

### 어느 기관이 명목 원금 1억 달러($100,000,000)에 대하여, 1.5년 후부터 2년 후 사이(6개월간)에 발생할 선도 SOFR 금리를 지급하고, 고정금리 연 5.8%를 수취하기로 하는 선도금리계약(FRA)을 체결하였다. 현재 시장에서 예측하는 해당 기간의 6개월 일별 복리(Daily Compounded) 선도 SOFR 금리가 연 5%라고 한다. 만기 2년인 무위험 SOFR 기준의 연속복리 할인율이 연 4%라고 할 때, 현재 시점에서 이 FRA 계약의 공정 가치는 얼마인가? (단, 이자 계산 일수 기준은 ACT/360이며, 해당 6개월 구간의 일수 비율은 정확히 0.5년으로 가정한다.)

In [2]:
from math import exp
N=100000000
N*(0.058-0.05)*0.5*exp(-0.04*2)

369246.5385546543

### 얼마 전 한 금융기관은 $1억 달러의 명목 원금에 대해 연 3%의 고정금리를 지급하고, 6개월물 SOFR 준거금리(변동금리)를 수취하기로 하는 이자율 스왑 계약을 체결하였다. 이 스왑의 잔존 만기는 현재 1.2년이다. 이에 따라 현금흐름(이자) 교환은 0.2년, 0.7년, 그리고 1.2년 후에 각각 이루어질 것이다. SOFR를 바탕으로 산출한 0.2년, 0.7년, 1.2년 만기의 연속복리 기준 무위험 금리가 각각 2.8%, 3.2%, 3.4%라고 가정하자. 또한, 지난 0.3년 동안(이 기간은 0.2년 후의 현금흐름 교환을 결정할 전체 일수의 60%를 포함함) 관측된 연속복리 기준 무위험 금리는 2.3%라고 가정하자. 0.2년 후 교환될 변동금리는 0.6×2.3% + 0.4×2.8%로 계산되어 2.50%가 된다. 0.7년 후 교환될 변동금리는 0.2년과 0.7년 사이 구간에 대한 선도금리(Forward Rate)이다. 이는 연속복리 기준으로 3.36%이다. 이와 유사하게, 1.2년 후 교환될 변동금리는 0.7년과 1.2년 사이 구간에 대한 선도금리이며, 연속복리 기준으로 3.68%이다. 여기서 설명한 IRS Pay 포지션의 가치는 얼마일까?

In [17]:
Rm=2*(exp(0.025*0.5)-1)
Rm

0.025156903081268833

In [18]:
N=100000000
PV1=N*(Rm-0.03)*0.5*exp(-0.028*0.2)
PV1

-240802.56870949865

In [19]:
Rm=2*(exp(0.0336*0.5)-1)
Rm

0.03388382720465222

In [20]:
PV2=N*(Rm-0.03)*0.5*exp(-0.032*0.7)
PV2

189889.83075335

In [21]:
Rm=2*(exp(0.0368*0.5)-1)
Rm

0.03714064608849865

In [22]:
PV3=N*(Rm-0.03)*0.5*exp(-0.034*1.2)
PV3

342758.5509749218

In [23]:
PV1+PV2+PV3

291845.8130187731

### 무위험 제로 금리 곡선(Risk-free Zero Curve)이 연속복리 기준 연 6%로 모든 만기에 대해 평탄하다고 가정하자. 보유자에게 5년 후 시작하여 3년 동안 이어지는 이자율 스왑(IRS) 계약에서 연 6.2%의 고정금리를 지급(Pay)할 수 있는 권리를 부여하는 스왑션(Swaption)이 있다. 스왑의 이자 교환은 6개월마다(Semiannually) 이루어지며, 명목 원금은 1억 달러이다. 현재 시장의 선도 스왑 금리(Forward Swap Rate)는 연속복리 기준으로 연 6.1%이며, 이는 6개월 복리 기준으로 환산하면 연 6.194%이다. 선도 스왑 금리의 변동성(Volatility)이 연 20%라고 할 때, 이 스왑션의 공정 가치를 구하시오.

In [24]:
def black76(F0: float, K: float, T: float, discT:float,
            r:float, sigma: float, callput='c'):
  import math
  from scipy.stats import norm

  sig_sqrtT = sigma * math.sqrt(T)
  d1 = (math.log(F0 / K) + 0.5 * sigma * sigma * T) / sig_sqrtT
  d2 = d1 - sig_sqrtT

  df = math.exp(-r * discT)

  if callput == 'c':
    price = df * (F0 * norm.cdf(d1) - K * norm.cdf(d2))
  elif callput == 'p':
    price = df * (K * norm.cdf(-d2) - F0 * norm.cdf(-d1))

  return price

In [26]:
F0 = 0.06194
K = 0.062
T = 5
discT = 1
r = 0.06
sigma = 0.20

swaption=black76(F0, K, T, discT, r, sigma, 'c')
swaption

np.float64(0.010298001356369357)